# IMDb Movie Chatbot — Case Study 2 (RAG + Multi-Agent)

**Full Name:** Shih Wei Wong

**Uplevel Email Address:** `shihwei.wong@gmail.com`

**Problem Statement:** IMDb Movie Chatbot - Conversational movie search and recommendations 


# Project Overview 

This case study positions the IMDb chatbot as an **AI-powered content discovery product**. The goal is to help users move from vague intent or fragmented preferences to confident movie choices through natural conversation, relevant retrieval, and clear recommendation reasoning. 

**Business Objective:** Build a conversational movie discovery assistant that reduces search friction, improves recommendation quality and increases user engagement by helping people find the right title faster and with more confidence. It combines semantic retrieval, personalization and strong user experience to improve discovery, retention and user trust. 

In this notebook, Part 1 focuses on creating the data foundation for that product: understanding the dataset, making explicit cleaning decisions, and preparing retrieval-ready features that support better recommendations and more engaging conversations in Part 2. 



# Part 1 Objectives 
1. Understand the dataset shape, schema and quality risks 
2. Make explicit, reproducible cleaning decisions (especially duplicates)
3. Build a reliable `clean_df` with `movie_id` and `search_text` for Part 2 embeddings/FAISS. 
4. Document insights that matter for product behaviour and retrieval quality 

## Data Understanding and Data Quality Checks

In this section, I first inspect the structure and quality of the dataset before moving into exploratory data analysis (EDA). The goal here is to identify practical issues such as missing values, duplicate records, and formatting problems that could affect retrieval quality later in the chatbot pipeline.

In [8]:
# Environment setup: support both Google Colab and local notebook execution.
from pathlib import Path

IN_COLAB = False

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Running in Google Colab')
    print('Google Drive mounted successfully')
except ImportError:
    print('Running locally (VS Code / Jupyter)')
    print('No Google Drive mount needed - using local files')

if IN_COLAB:
    DATA_PATH = Path('/content/drive/MyDrive/colab-data/imdb_dataset.csv')
    PROJECT_ROOT = DATA_PATH.parent
else:
    PROJECT_ROOT = Path.cwd()
    DATA_PATH = PROJECT_ROOT / 'imdb_dataset.csv'

NOTEBOOK_PATH = PROJECT_ROOT / 'wei-wong.ipynb'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at: {DATA_PATH}')

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_PATH: {DATA_PATH}')

Running locally (VS Code / Jupyter)
No Google Drive mount needed - using local files
PROJECT_ROOT: /Users/weiwong/Documents/personal/projects/courses/2025-09-ik-ml-engineering/2026-03-08-case-study-2-imdb
DATA_PATH: /Users/weiwong/Documents/personal/projects/courses/2025-09-ik-ml-engineering/2026-03-08-case-study-2-imdb/imdb_dataset.csv


In [7]:
# Import libraries used in Part 1.
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display settings
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160) 

In [10]:
# Load IMDb dataset 
raw_df = pd.read_csv(DATA_PATH) 

print(f"Loaded dataset from: {DATA_PATH.resolve()}")
print(f"Shape: {raw_df.shape}")

raw_df.head()



Loaded dataset from: /Users/weiwong/Documents/personal/projects/courses/2025-09-ik-ml-engineering/2026-03-08-case-study-2-imdb/imdb_dataset.csv
Shape: (3173, 10)


,Title,IMDb Rating,Year,Certificates,Genre,Director,Star Cast,MetaScore,Poster-src,Duration (minutes)
0,End of the Spear,6.8,2005,PG-13,Adventure,Jim Hanon,Louie LeonardoChad AllenJack Guzman,45.0,https://m.media-amazon.com/images/M/MV5BMTYxOT...,108.0
1,Elvira Madigan,7.0,1967,PG,Biography,Bo Widerberg,Pia DegermarkThommy BerggrenLennart Malmer,66.0,https://m.media-amazon.com/images/M/MV5BMmY2Nj...,91.0
2,The Kid Stays in the Picture,7.3,2002,R,Documentary,Nanette Burstein,Robert EvansEddie AlbertPeter Bart,75.0,https://m.media-amazon.com/images/M/MV5BZjhiZm...,93.0
3,It Ain't Over,8.2,2022,PG,Documentary,Sean Mullin,Andy AndresRoger AngellMarty Appel,79.0,https://m.media-amazon.com/images/M/MV5BZWViYW...,99.0
4,Mahler,7.0,1974,PG,Biography,Ken Russell,Robert PowellGeorgina HaleLee Montague,66.0,https://m.media-amazon.com/images/M/MV5BYzY4Mz...,115.0


**Insights:**
The dataset contains 3,173 rows and 10 columns - a good size for building a lightweight movie retrieval system. 

The columns are relevant for recommendation and filtering, but the `Star Cast` column appears messy because actor names are concatenated together. This will need cleaning before cast information can be used reliably.

In [11]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3173 entries, 0 to 3172
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               3173 non-null   object 
 1   IMDb Rating         3173 non-null   float64
 2   Year                3173 non-null   int64  
 3   Certificates        3173 non-null   object 
 4   Genre               3173 non-null   object 
 5   Director            3173 non-null   object 
 6   Star Cast           3173 non-null   object 
 7   MetaScore           3173 non-null   float64
 8   Poster-src          3173 non-null   object 
 9   Duration (minutes)  3173 non-null   float64
dtypes: float64(3), int64(1), object(6)
memory usage: 248.0+ KB


In [13]:
raw_df.isna().sum()

Title                 0
IMDb Rating           0
Year                  0
Certificates          0
Genre                 0
Director              0
Star Cast             0
MetaScore             0
Poster-src            0
Duration (minutes)    0
dtype: int64

There are no missing values in the dataset, which is a good sign for downstream retrieval and filtering. The main data quality issue appears to be formatting consistency, especially in the Star Cast column, rather than null handling.

In [22]:
# Build a focused data-quality summary for issues that affect cleaning decisions.
assert "raw_df" in globals(), "Run the dataset loading cell first so raw_df is available."

def build_data_quality_snapshot(df: pd.DataFrame) -> pd.DataFrame:
    cast_text = df["Star Cast"].fillna("").astype(str)
    cast_has_delimiter = cast_text.str.contains(r",|/|;|\|")

    snapshot_rows = [
        {"Metric": "Exact duplicate rows", "Value": int(df.duplicated().sum())},
        {"Metric": "Duplicate titles", "Value": int(df["Title"].duplicated().sum())},
        {"Metric": "Duplicate title-year pairs", "Value": int(df.duplicated(subset=["Title", "Year"]).sum())},
        {"Metric": "Rows with clearly delimited cast", "Value": int(cast_has_delimiter.sum())},
        {"Metric": "Rows with non-delimited cast", "Value": int((~cast_has_delimiter).sum())},
        {"Metric": "Percent non-delimited cast", "Value": round((~cast_has_delimiter).mean() * 100, 2)},
    ]

    return pd.DataFrame(snapshot_rows)

data_quality_snapshot = build_data_quality_snapshot(raw_df)
display(data_quality_snapshot)

,Metric,Value
0,Exact duplicate rows,411.0
1,Duplicate titles,434.0
2,Duplicate title-year pairs,413.0
3,Rows with clearly delimited cast,0.0
4,Rows with non-delimited cast,3173.0
5,Percent non-delimited cast,100.0


**Comments:**

The dataset contains 411 exact duplicate rows, which suggests that some records have been repeated without any changes across the full set of columns. Those are usually safe to remove because they do not add any new information.

There are also 413 duplicate `Title + Year` pairs, which is slightly higher than the number of exact duplicate rows. This tells me that a small number of movies share the same title and release year but differ in other fields, so I should inspect those cases before deciding whether to remove them automatically.

The `Star Cast` field remains a separate cleaning issue. Since many actor names are concatenated together, I will handle that later as its own step rather than mixing it into the duplicate-cleaning decision.

**Duplicate Handling Decision:**

Based on this inspection, I will treat exact duplicate rows and duplicate `Title + Year` pairs differently. Exact duplicate rows can be removed safely because they repeat the same movie record without adding new information. In contrast, duplicate `Title + Year` pairs need closer review because some may differ in metadata rather than being true copy-paste duplicates.

My cleaning decision is therefore: remove exact duplicate rows first, then inspect any remaining duplicate `Title + Year` cases before creating a final movie identifier. This is a safer approach than dropping all repeated titles automatically, because it reduces the risk of removing useful records by mistake.

In [28]:
# Inspect duplicate records more closely.
exact_duplicate_rows = raw_df[raw_df.duplicated(keep=False)].sort_values(["Title", "Year"])

title_year_duplicate_rows = raw_df[
    raw_df.duplicated(subset=["Title", "Year"], keep=False)
].sort_values(["Title", "Year"])

non_exact_title_year_duplicates = title_year_duplicate_rows.groupby(["Title", "Year"], group_keys=False).filter(
    lambda group: len(group.drop_duplicates()) > 1
)

print(f"Duplicate rows that would be removed if I keep the first copy: {raw_df.duplicated().sum()}")
print(f"Rows shown below that belong to exact-duplicate groups: {len(exact_duplicate_rows)}")
print(f"Rows shown below that belong to duplicate Title + Year groups: {len(title_year_duplicate_rows)}")
print(f"Rows in duplicate Title + Year groups that are not exact full-row duplicates: {len(non_exact_title_year_duplicates)}")

display(exact_duplicate_rows.head(10))

display(
    non_exact_title_year_duplicates[[
        "Title",
        "Year",
        "IMDb Rating",
        "Certificates",
        "Genre",
        "Director",
        "Star Cast",
        "MetaScore",
        "Duration (minutes)",
    ]].head(10)
)

Duplicate rows that would be removed if I keep the first copy: 411
Rows shown below that belong to exact-duplicate groups: 782
Rows shown below that belong to duplicate Title + Year groups: 784
Rows in duplicate Title + Year groups that are not exact full-row duplicates: 6


,Title,IMDb Rating,Year,Certificates,Genre,Director,Star Cast,MetaScore,Poster-src,Duration (minutes)
686,12 Years a Slave,8.1,2013,R,Biography,Steve McQueen,Chiwetel EjioforMichael Kenneth WilliamsMichae...,96.0,https://m.media-amazon.com/images/M/MV5BMjExMT...,134.0
2886,12 Years a Slave,8.1,2013,R,Biography,Steve McQueen,Chiwetel EjioforMichael Kenneth WilliamsMichae...,96.0,https://m.media-amazon.com/images/M/MV5BMjExMT...,134.0
360,1492: Conquest of Paradise,6.4,1992,PG-13,Adventure,Ridley Scott,Gérard DepardieuArmand AssanteSigourney Weaver,47.0,https://m.media-amazon.com/images/M/MV5BNmVlMW...,154.0
2702,1492: Conquest of Paradise,6.4,1992,PG-13,Adventure,Ridley Scott,Gérard DepardieuArmand AssanteSigourney Weaver,47.0,https://m.media-amazon.com/images/M/MV5BNmVlMW...,154.0
49,20 Feet from Stardom,7.4,2013,PG-13,Documentary,Morgan Neville,Darlene LoveMerry ClaytonLisa Fischer,83.0,https://m.media-amazon.com/images/M/MV5BMTQxND...,91.0
2499,20 Feet from Stardom,7.4,2013,PG-13,Documentary,Morgan Neville,Darlene LoveMerry ClaytonLisa Fischer,83.0,https://m.media-amazon.com/images/M/MV5BMTQxND...,91.0
412,22 July,6.8,2018,R,Biography,Paul Greengrass,Anders Danielsen LieJonas Strand GravliJon Øig...,69.0,https://m.media-amazon.com/images/M/MV5BNGY2Mm...,143.0
2730,22 July,6.8,2018,R,Biography,Paul Greengrass,Anders Danielsen LieJonas Strand GravliJon Øig...,69.0,https://m.media-amazon.com/images/M/MV5BNGY2Mm...,143.0
864,A Haunting in Venice,6.5,2023,PG-13,Crime,Kenneth Branagh,Kenneth BranaghMichelle YeohJamie Dornan,63.0,https://m.media-amazon.com/images/M/MV5BYmI4Mz...,103.0
914,A Haunting in Venice,6.5,2023,PG-13,Crime,Kenneth Branagh,Kenneth BranaghMichelle YeohJamie Dornan,63.0,https://m.media-amazon.com/images/M/MV5BYmI4Mz...,103.0


,Title,Year,IMDb Rating,Certificates,Genre,Director,Star Cast,MetaScore,Duration (minutes)
1606,Sacrifice,2010,6.2,R,Sci-Fi,Bob Nelson,Brent HeffronShanda Lee MunsonHeather Liebenow,66.0,116.3
2270,Sacrifice,2010,6.2,R,Action,Kaige Chen,You GeXueqi WangXiaoming Huang,57.0,122.0
85,The Message,1976,8.9,PG,Action,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Rahman El-Shar...,66.0,207.0
406,The Message,1976,8.1,PG,Biography,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Jawdat El-Sahhar,66.0,177.0
2532,The Message,1976,8.9,PG,Action,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Rahman El-Shar...,66.0,207.0
2727,The Message,1976,8.1,PG,Biography,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Jawdat El-Sahhar,66.0,177.0


## Prepare a Clean Base Table for Retrieval

Before moving into EDA, I remove the most obvious structural issues that would distort counts and later retrieval quality. That means removing exact duplicate rows first, then checking what duplicate `Title + Year` cases still remain.

More retrieval-specific feature engineering, such as cast parsing and building `search_text`, will come later after I understand the cleaned dataset better.

In [30]:
# Remove exact duplicate rows and create a working copy for further cleaning.
clean_df = raw_df.drop_duplicates().copy()

remaining_title_year_duplicates = clean_df[
    clean_df.duplicated(subset=["Title", "Year"], keep=False)
].sort_values(["Title", "Year"])

print(f"Rows before removing exact duplicates: {len(raw_df)}")
print(f"Rows after removing exact duplicates: {len(clean_df)}")
print(f"Exact duplicate rows removed: {len(raw_df) - len(clean_df)}")
print(f"Rows still involved in duplicate Title + Year groups: {len(remaining_title_year_duplicates)}")

display(
    remaining_title_year_duplicates[[
        "Title",
        "Year",
        "IMDb Rating",
        "Certificates",
        "Genre",
        "Director",
        "Star Cast",
        "MetaScore",
        "Duration (minutes)",
    ]].head(10)
)

Rows before removing exact duplicates: 3173
Rows after removing exact duplicates: 2762
Exact duplicate rows removed: 411
Rows still involved in duplicate Title + Year groups: 4


,Title,Year,IMDb Rating,Certificates,Genre,Director,Star Cast,MetaScore,Duration (minutes)
1606,Sacrifice,2010,6.2,R,Sci-Fi,Bob Nelson,Brent HeffronShanda Lee MunsonHeather Liebenow,66.0,116.3
2270,Sacrifice,2010,6.2,R,Action,Kaige Chen,You GeXueqi WangXiaoming Huang,57.0,122.0
85,The Message,1976,8.9,PG,Action,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Rahman El-Shar...,66.0,207.0
406,The Message,1976,8.1,PG,Biography,Moustapha Akkad,H.A.L. CraigTewfik El-HakimA.B. Jawdat El-Sahhar,66.0,177.0


**Cleaning Decision:**

I remove exact duplicate rows before EDA because they can distort counts, distributions, and later retrieval results. They repeat the same movie record without adding any new information, so dropping them is a low-risk cleaning step.

I do not automatically remove the remaining duplicate `Title + Year` cases. After exact deduplication, I inspected the small number of remaining collisions manually. In the case of `Sacrifice`, the rows refer to two different films: one is a Chinese historical drama directed by Chen Kaige, while the other is a science fiction film. Those should both be kept.

For `The Message` (1976), the rows are more ambiguous. They share the same director, writers, and Metascore, but differ in IMDb rating, genre, running time, and cast. Since the source data is inconsistent and I cannot justify collapsing them into one record with confidence, I will keep both rows for now rather than risk deleting a potentially valid record.

Because of these remaining collisions, I created a `movie_id` so each movie row has a stable unique identifier. This is important because titles can repeat, and in a few cases even `Title + Year` is not enough to distinguish records safely. Using `movie_id` reduces ambiguity and makes the retrieval pipeline more reliable.

In [32]:
# Create a stable movie_id for retrieval.
def slugify_title(title: str) -> str:
    title = str(title).strip().lower()
    title = re.sub(r"[^a-z0-9]+", "_", title)
    title = re.sub(r"_+", "_", title).strip("_")
    return title

clean_df["base_movie_id"] = (
    clean_df["Title"].apply(slugify_title) + "_" + clean_df["Year"].astype(int).astype(str)
)

movie_id_suffix = clean_df.groupby("base_movie_id").cumcount()

clean_df["movie_id"] = np.where(
    movie_id_suffix == 0,
    clean_df["base_movie_id"],
    clean_df["base_movie_id"] + "_" + (movie_id_suffix + 1).astype(str),
)

print(f"Unique movie_id values: {clean_df['movie_id'].nunique()} / {len(clean_df)}")
display(clean_df[["Title", "Year", "movie_id"]].head(10))

Unique movie_id values: 2762 / 2762


,Title,Year,movie_id
0,End of the Spear,2005,end_of_the_spear_2005
1,Elvira Madigan,1967,elvira_madigan_1967
2,The Kid Stays in the Picture,2002,the_kid_stays_in_the_picture_2002
3,It Ain't Over,2022,it_ain_t_over_2022
4,Mahler,1974,mahler_1974
5,The Dark Horse,2014,the_dark_horse_2014
6,Stephen Curry: Underrated,2023,stephen_curry_underrated_2023
7,Carrington,1995,carrington_1995
8,Burden,2018,burden_2018
9,Georgetown,2019,georgetown_2019


## Exploratory Data Analysis (EDA)

With exact duplicate rows removed and a stable `movie_id` assigned, I can now explore the dataset on a cleaner base table. In this section, I will look at patterns such as year coverage, rating distributions, genres, and runtime so I can understand what kind of movie catalog this chatbot is working with before building retrieval features like `search_text`.